In [32]:
import os

print(os.path.exists("../outputs/best_xgboost_model.pkl"))
print(os.path.exists("../outputs/model_features.json"))

print(os.listdir("../outputs"))

True
True
['best_model_params.json', 'best_xgboost_model.pkl', 'customer_risk_scores.csv', 'model_comparison.csv', 'model_features.json', 'retention_strategy_comparison.csv', 'shap_feature_importance.png']


In [31]:
import joblib
import json
import os

# Make sure outputs folder exists
os.makedirs("../outputs", exist_ok=True)

# Save the best XGBoost model
joblib.dump(
    search.best_estimator_,
    "../outputs/best_xgboost_model.pkl"
)

# Save the exact feature columns used during training
with open(
    "../outputs/model_features.json",
    "w"
) as f:
    json.dump(
        X.columns.tolist(),
        f,
        indent=2
    )

print("Model saved successfully!")
print("Feature list saved successfully!")
print("Number of features:", len(X.columns))

Model saved successfully!
Feature list saved successfully!
Number of features: 38


In [29]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_cv = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

rf_cv = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

xgb_cv = cross_val_score(
    xgb_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("Logistic:", logistic_cv.mean(), "±", logistic_cv.std())
print("Random Forest:", rf_cv.mean(), "±", rf_cv.std())
print("XGBoost:", xgb_cv.mean(), "±", xgb_cv.std())


model_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "ROC_AUC": logistic_cv.mean(),
        "ROC_AUC_STD": logistic_cv.std()
    },
    {
        "Model": "Random Forest",
        "ROC_AUC": rf_cv.mean(),
        "ROC_AUC_STD": rf_cv.std()
    },
    {
        "Model": "XGBoost",
        "ROC_AUC": xgb_cv.mean(),
        "ROC_AUC_STD": xgb_cv.std()
    }
])

model_results.to_csv(
    "../outputs/model_comparison.csv",
    index=False
)

print(model_results)

model_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "ROC_AUC": logistic_cv.mean(),
        "ROC_AUC_STD": logistic_cv.std()
    },
    {
        "Model": "Random Forest",
        "ROC_AUC": rf_cv.mean(),
        "ROC_AUC_STD": rf_cv.std()
    },
    {
        "Model": "XGBoost",
        "ROC_AUC": xgb_cv.mean(),
        "ROC_AUC_STD": xgb_cv.std()
    }
])

model_results.to_csv(
    "../outputs/model_comparison.csv",
    index=False
)

print(model_results)

Logistic: 0.8472735395491003 ± 0.004160556911806833
Random Forest: 0.8293587534807857 ± 0.004642883437568931
XGBoost: 0.8436050311874095 ± 0.003614870588697133
                 Model   ROC_AUC  ROC_AUC_STD
0  Logistic Regression  0.847274     0.004161
1        Random Forest  0.829359     0.004643
2              XGBoost  0.843605     0.003615
                 Model   ROC_AUC  ROC_AUC_STD
0  Logistic Regression  0.847274     0.004161
1        Random Forest  0.829359     0.004643
2              XGBoost  0.843605     0.003615


In [30]:
from sklearn.model_selection import RandomizedSearchCV
import json

best_params = search.best_params_

with open(
    "../outputs/best_model_params.json",
    "w"
) as f:
    json.dump(
        best_params,
        f,
        indent=4
    )

print("Best parameters saved:")
print(best_params)

params = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

search = RandomizedSearchCV(
    xgb_model,
    params,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)

print("Best CV ROC-AUC:")
print(search.best_score_)

best_model = search.best_estimator_

final_results = evaluate_model(
    best_model,
    X_test,
    y_test
)

final_results



Best parameters saved:
{'subsample': 0.7, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
Best parameters:
{'subsample': 0.7, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
Best CV ROC-AUC:
0.8484864803582564


{'Accuracy': 0.798862828713575,
 'Precision': 0.6501650165016502,
 'Recall': 0.5267379679144385,
 'F1': 0.5819793205317577,
 'ROC_AUC': 0.8415147718860492,
 'PR_AUC': 0.6530851222334262}

In [18]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
logistic_cv = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

print(
    f"Logistic Regression ROC-AUC: "
    f"{logistic_cv.mean():.3f} ± {logistic_cv.std():.3f}"
)
rf_cv = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print(
    f"Random Forest ROC-AUC: "
    f"{rf_cv.mean():.3f} ± {rf_cv.std():.3f}"
)
xgb_cv = cross_val_score(
    xgb_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print(
    f"XGBoost ROC-AUC: "
    f"{xgb_cv.mean():.3f} ± {xgb_cv.std():.3f}"
)

Logistic Regression ROC-AUC: 0.847 ± 0.004
Random Forest ROC-AUC: 0.829 ± 0.005
XGBoost ROC-AUC: 0.844 ± 0.004


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

rf_results = evaluate_model(
    rf_model,
    X_test,
    y_test
)

rf_results


{'Accuracy': 0.7846481876332623,
 'Precision': 0.6219931271477663,
 'Recall': 0.4839572192513369,
 'F1': 0.544360902255639,
 'ROC_AUC': 0.8226661869535282,
 'PR_AUC': 0.608988935464578}

In [15]:


from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)


xgb_results = evaluate_model(
    xgb_model,
    X_test,
    y_test
)

xgb_results

{'Accuracy': 0.7803837953091685,
 'Precision': 0.6025236593059937,
 'Recall': 0.5106951871657754,
 'F1': 0.552821997105644,
 'ROC_AUC': 0.831297140875183,
 'PR_AUC': 0.6401933628921008}

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
logistic_model.fit(
    X_train,
    y_train
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_model(model, X_test, y_test):

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "Precision": precision_score(
            y_test,
            predictions
        ),
        "Recall": recall_score(
            y_test,
            predictions
        ),
        "F1": f1_score(
            y_test,
            predictions
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            probabilities
        ),
        "PR_AUC": average_precision_score(
            y_test,
            probabilities
        )
    }
logistic_results = evaluate_model(
    logistic_model,
    X_test,
    y_test
)

logistic_results



{'Accuracy': 0.7803837953091685,
 'Precision': 0.6025236593059937,
 'Recall': 0.5106951871657754,
 'F1': 0.552821997105644,
 'ROC_AUC': 0.831297140875183,
 'PR_AUC': 0.6401933628921008}

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv(
    "../data/processed/customer_master.csv"
)
X_raw = data.drop(
    columns=[
        "customerID",
        "Churn",
        "Churn_Target"
    ]
)

y = data["Churn_Target"]

X = pd.get_dummies(
    X_raw,
    drop_first=True
)

X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)
